In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import erfinv, erf

NL = chr(10)

# 1. Загрузка 50 губерний Европейской России (1904, Lindert & Nafziger)
xl_1904 = pd.ExcelFile("1904_Medium-ineq_distrib_14dec'11.xlsx")
d_m = xl_1904.parse("(8) Results, M-estimates")

SUMMARY_LABELS = {'max', 'min', 'median', 'mean', 'std', 'st. dev.', 'cv', 'total', 'average', 'european russia'}
rows_1904 = []
for _, row in d_m.iloc[4:].iterrows():
    pname = str(next(iter(row))).strip()
    if not pname or pname.lower() in SUMMARY_LABELS or any(w in pname.lower() for w in ['total', 'european', 'mean', 'median']):
        continue
    vals = [pd.to_numeric(v, errors='coerce') for v in row.iloc[1:16]]
    if not np.isnan(np.sum(vals)):
        rows_1904.append([pname] + vals)

cols_m = [
    'Province', 'Gini', 'Mean_Inc', 'Median_Inc',
    'Top1_Share', 'Top1_Avg', 'Top5_Share', 'Top5_Avg',
    'Top10_Share', 'Top10_Avg', 'Top20_Share', 'Top20_Avg',
    'Next40_Share', 'Next40_Avg', 'Bot40_Share', 'Bot40_Avg'
]
df_1904 = pd.DataFrame(rows_1904, columns=cols_m).iloc[:50].copy()
df_1904['Mean_Median_Ratio'] = df_1904['Mean_Inc'] / df_1904['Median_Inc']

# 2. Загрузка 87 субъектов РФ (2025, Росстат ЕМИСС)
d_g = pd.ExcelFile('gini.xls').parse('Данные').iloc[2:].copy()
d_g.columns = ['Region', 'unit', 'year', 'h1_2025', 'Gini_2025']
d_g['Region'] = d_g['Region'].astype(str).str.strip()
d_g['Gini_2025'] = pd.to_numeric(d_g['Gini_2025'], errors='coerce')
d_g = d_g.dropna(subset=['Gini_2025'])
d_g = d_g[~d_g['Region'].str.contains('Российская Федерация|федеральный округ', case=False, na=False)]

d_inc = pd.ExcelFile('mean_inc.xls').parse('Данные').iloc[3:].copy()
d_inc_clean = pd.DataFrame({
    'Region': d_inc.iloc[:, 0].astype(str).str.strip(),
    'q1_25': pd.to_numeric(d_inc.iloc[:, 5], errors='coerce'),
    'q2_25': pd.to_numeric(d_inc.iloc[:, 6], errors='coerce'),
    'q3_25': pd.to_numeric(d_inc.iloc[:, 7], errors='coerce'),
    'q4_25': pd.to_numeric(d_inc.iloc[:, 8], errors='coerce'),
})
d_inc_clean['Mean_Inc_2025'] = d_inc_clean[['q1_25', 'q2_25', 'q3_25', 'q4_25']].mean(axis=1)
d_inc_clean = d_inc_clean.dropna(subset=['Mean_Inc_2025'])

df_2025 = pd.merge(d_g[['Region', 'Gini_2025']], d_inc_clean[['Region', 'Mean_Inc_2025']], on='Region', how='inner')
df_2025 = df_2025[~df_2025['Region'].str.contains('без автономн', case=False, na=False)].drop_duplicates(subset=['Region']).copy()

def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))

sigma_25 = 2.0 * erfinv(df_2025['Gini_2025'].to_numpy(dtype=float))
df_2025['Mean_Median_Ratio'] = np.exp(0.5 * sigma_25**2)
df_2025['Median_Inc_2025'] = df_2025['Mean_Inc_2025'] / df_2025['Mean_Median_Ratio']
df_2025['Bot40_Share'] = norm_cdf(-0.2533471 - sigma_25) * 100.0
df_2025['Top20_Share'] = (1.0 - norm_cdf(0.8416212 - sigma_25)) * 100.0

SHORT_NAMES = {
    'Город Москва столица Российской Федерации город федерального значения': 'Moscow',
    'Город Санкт-Петербург город федерального значения': 'St. Petersburg',
    'Ямало-Ненецкий автономный округ (Тюменская область)': 'Yamalo-Nenets AO',
    'Чукотский автономный округ': 'Chukotka AO',
    'Ненецкий автономный округ (Архангельская область)': 'Nenets AO',
    'Тюменская область': 'Tyumen Oblast',
    'Республика Саха (Якутия)': 'Sakha (Yakutia)',
    'Магаданская область': 'Magadan Oblast',
    'Республика Ингушетия': 'Ingushetia',
    'Город федерального значения Севастополь': 'Sevastopol',
    'Республика Крым': 'Crimea',
    'Карачаево-Черкесская Республика': 'Karachay-Cherkessia',
    'Московская область': 'Moscow Oblast',
    'Ленинградская область': 'Leningrad Oblast',
    'Сахалинская область': 'Sakhalin Oblast',
    'Республика Татарстан (Татарстан)': 'Tatarstan',
    'Свердловская область': 'Sverdlovsk Oblast'
}
df_2025['ShortName'] = df_2025['Region'].map(SHORT_NAMES).fillna(df_2025['Region'])

ASIAN_KEYWORDS = ['Урал', 'Тюмен', 'Ямало', 'Ханты', 'Курган', 'Свердлов', 'Челябин', 'Сибир', 'Алтай', 'Тыва', 'Хакас', 'Бурят', 'Забайкал', 'Краснояр', 'Иркут', 'Кемеров', 'Новосибир', 'Омск', 'Томск', 'Саха', 'Камчат', 'Примор', 'Хабаров', 'Амур', 'Магадан', 'Сахалин', 'Еврей', 'Чукот']
is_asian = df_2025['Region'].apply(lambda s: any(k.lower() in s.lower() for k in ASIAN_KEYWORDS))
df_2025_eur = df_2025[~is_asian].copy()

# 3. Спектральный аудит и бутстрэп-интервалы (Таблица 1)
def full_audit(df_in, name_col, feat_cols, label):
    X = df_in[feat_cols].to_numpy(dtype=float)
    Z = (X - np.mean(X, axis=0)) / np.std(X, axis=0, ddof=1)
    N_obs, K_obs = Z.shape
    U, s, Vt = np.linalg.svd(Z, full_matrices=False)
    u1, u2 = np.take(U, 0, axis=1), np.take(U, 1, axis=1)
    v1, v2 = np.take(Vt, 0, axis=0), np.take(Vt, 1, axis=0)
    if v1.item(0) < 0:
        u1, v1 = -u1, -v1
    if v2.item(2) < 0:
        u2, v2 = -u2, -v2
    var1 = (s.item(0)**2) / np.sum(s**2) * 100.0
    var2 = (s.item(1)**2) / np.sum(s**2) * 100.0
    pr_u = (1.0 / np.sum(u1**4) - 1.0) / (N_obs - 1.0)
    pr_v = (1.0 / np.sum(v1**4) - 1.0) / (K_obs - 1.0)

    rng = np.random.default_rng(42)
    sub_n = int(np.floor(N_obs * 0.8))
    bu = []
    for _ in range(300):
        idx = np.sort(rng.choice(N_obs, size=sub_n, replace=False))
        Xb = X[idx, :]
        Zb = (Xb - np.mean(Xb, axis=0)) / np.std(Xb, axis=0, ddof=1)
        Ub, _, _ = np.linalg.svd(Zb, full_matrices=False)
        u1b = np.take(Ub, 0, axis=1)
        bu.append((1.0 / np.sum(u1b**4) - 1.0) / (sub_n - 1.0))

    en = pd.Series((u1**2)*100.0, index=df_in[name_col]).sort_values(ascending=False)
    print(f"{label:<32} | N={N_obs:2d} | Var1={var1:4.1f}% | PR_reg={pr_u:.3f} [{np.percentile(bu, 2.5):.3f}, {np.percentile(bu, 97.5):.3f}] | PR_feat={pr_v:.3f} | Top2={en.head(2).sum():.1f}%")
    return u1, u2, var1, var2, pr_u, pr_v

print("=== TWO-CENTURY SPATIAL DUALISM AUDIT (1904 vs 2025) ===")
u1_04, u2_04, v1_04, v2_04, pru_04, prv_04 = full_audit(df_1904, 'Province', ['Gini', 'Mean_Inc', 'Median_Inc', 'Top20_Share', 'Bot40_Share', 'Mean_Median_Ratio'], "Imperial Russia (1904)")
u1_25, u2_25, v1_25, v2_25, pru_25, prv_25 = full_audit(df_2025, 'ShortName', ['Gini_2025', 'Mean_Inc_2025', 'Median_Inc_2025', 'Top20_Share', 'Bot40_Share', 'Mean_Median_Ratio'], "Modern Russia All (2025)")
u1_25e, u2_25e, v1_25e, v2_25e, pru_25e, prv_25e = full_audit(df_2025_eur, 'ShortName', ['Gini_2025', 'Mean_Inc_2025', 'Median_Inc_2025', 'Top20_Share', 'Bot40_Share', 'Mean_Median_Ratio'], "Modern European RF (2025)")

df_1904['u1'], df_1904['u2'], df_1904['energy_%'] = u1_04, u2_04, (u1_04**2)*100.0
df_2025['u1'], df_2025['u2'], df_2025['energy_%'] = u1_25, u2_25, (u1_25**2)*100.0

# 4. Отрисовка публикационных графиков (300 dpi)
fig1 = plt.figure(figsize=(15.5, 10.5), dpi=300)
plt.subplots_adjust(hspace=0.38, wspace=0.26)

ax1 = fig1.add_subplot(2, 2, 1)
top10_04 = df_1904.sort_values(by='energy_%', ascending=False).head(10)
cols_04 = ['#c00000' if p in ['Sankt-Peterburg', 'Moskva'] else '#1f4e78' for p in top10_04['Province']]
x10 = np.arange(10)
ax1.bar(x10, top10_04['energy_%'], color=cols_04)
ax1.axhline(100.0 / 50.0, color='black', linestyle=':', lw=1.1, label='Uniform Benchmark (1/50 = 2.0%)')
ax1.text(2.5, 33.5, 'Two Capitals = 63.9%' + NL + 'of All Mode 1 Energy', ha='center', fontsize=8.5, fontweight='bold', color='#800000')
ax1.set_xticks(x10)
ax1.set_xticklabels(top10_04['Province'], rotation=38, ha='right', fontsize=8.2)
ax1.set_ylim(0, 42)
ax1.set_ylabel("Share of Mode 1 Spatial Energy (%)", fontsize=9.5)
ax1.set_title(f"A. 1904 Imperial Russia (N = 50): Capital Monopoly (PR_reg = {pru_04:.3f})", fontsize=10.2, fontweight='bold')
ax1.legend(loc='upper right', fontsize=8)
ax1.grid(True, axis='y', linestyle='--', alpha=0.4)

ax2 = fig1.add_subplot(2, 2, 2)
top10_25 = df_2025.sort_values(by='energy_%', ascending=False).head(10)
RES_SET = {'Chukotka AO', 'Yamalo-Nenets AO', 'Nenets AO', 'Tyumen Oblast', 'Magadan Oblast', 'Sakha (Yakutia)'}
CAP_SET = {'Moscow', 'St. Petersburg'}
cols_25 = ['#c00000' if p in CAP_SET else ('#ed7d31' if p in RES_SET else '#1f4e78') for p in top10_25['ShortName']]
ax2.bar(x10, top10_25['energy_%'], color=cols_25)
ax2.axhline(100.0 / len(df_2025), color='black', linestyle=':', lw=1.1, label=f'Uniform Benchmark (1/{len(df_2025)} = {100.0/len(df_2025):.2f}%)')
ax2.text(4.2, 13.0, 'Arctic Resource Hubs + Moscow' + NL + 'Top 5 = 50.9% (PR_reg = 0.161)', ha='center', fontsize=8.3, fontweight='bold', color='#800000')
ax2.set_xticks(x10)
ax2.set_xticklabels(top10_25['ShortName'], rotation=38, ha='right', fontsize=8.2)
ax2.set_ylim(0, 17.5)
ax2.set_ylabel("Share of Mode 1 Spatial Energy (%)", fontsize=9.5)
ax2.set_title(f"B. 2025 Russian Federation (N = {len(df_2025)}): Resource-Capital Archipelago", fontsize=10.2, fontweight='bold')
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(True, axis='y', linestyle='--', alpha=0.4)

ax3 = fig1.add_subplot(2, 2, 3)
for _, r in df_2025.iterrows():
    sn = r['ShortName']
    xv, yv = float(r['u1']), float(r['u2'])
    cpt = '#c00000' if sn in CAP_SET else ('#ed7d31' if sn in RES_SET else ('#385723' if sn in {'Ingushetia', 'Sevastopol', 'Crimea', 'Karachay-Cherkessia'} else '#595959'))
    sz = 105 if sn in CAP_SET else (85 if sn in RES_SET else 45)
    ax3.scatter([xv], [yv], color=cpt, s=sz, alpha=0.85, zorder=5)

lbl_25_map = [
    ('Chukotka AO', -0.11, 0.03),
    ('Yamalo-Nenets AO', -0.14, -0.055),
    ('Nenets AO', -0.10, 0.03),
    ('Moscow', -0.08, -0.055),
    ('St. Petersburg', 0.015, -0.045),
    ('Ingushetia', 0.015, -0.025),
    ('Sevastopol', 0.015, 0.02)
]
for sn, dx, dy in lbl_25_map:
    sub_r = df_2025[df_2025['ShortName'] == sn]
    if len(sub_r) > 0:
        xv = float(next(iter(sub_r['u1'])))
        yv = float(next(iter(sub_r['u2'])))
        ax3.annotate(sn, xy=(xv, yv), xytext=(xv + dx, yv + dy), fontsize=8.0, fontweight='bold' if sn in ['Moscow', 'Chukotka AO', 'Yamalo-Nenets AO', 'Ingushetia'] else 'normal', color='#222222')

ax3.axhline(0, color='black', linestyle=':', lw=0.9)
ax3.axvline(0, color='black', linestyle=':', lw=0.9)
ax3.set_xlim(-0.24, 0.44)
ax3.set_ylim(-0.28, 0.44)
ax3.set_xlabel(f"Mode 1 Income & Inequality Score (u1, Var1 = {v1_25:.1f}%)", fontsize=9.5)
ax3.set_ylabel(f"Mode 2 Orthogonal Skewness Score (u2, Var2 = {v2_25:.1f}%)", fontsize=9.5)
ax3.set_title("C. Structural Topology of 87 Russian Regions (2025)", fontsize=10.2, fontweight='bold')
ax3.grid(True, linestyle='--', alpha=0.4)
ax4 = fig1.add_subplot(2, 2, 4)
ax4.hist(df_1904['Gini'], bins=14, alpha=0.65, color='#1f4e78', edgecolor='black', label='1904 Provinces (N=50): Median=0.312, Max=0.611')
ax4.hist(df_2025['Gini_2025'], bins=14, alpha=0.65, color='#c00000', edgecolor='black', label='2025 Regions (N=87): Median=0.382, Max=0.470')
ax4.set_xlabel("Regional Gini Coefficient", fontsize=9.5)
ax4.set_ylabel("Number of Regions / Provinces", fontsize=9.5)
ax4.set_title("D. Shift in Regional Gini Distributions: 1904 vs. 2025", fontsize=10.2, fontweight='bold')
ax4.legend(loc='upper right', fontsize=7.8)
ax4.grid(True, axis='y', linestyle='--', alpha=0.4)
fig1.suptitle("Two Centuries of Russian Regional Dualism: Spatial Anatomy of Income Inequality in 1904 vs. 2025", fontsize=12.0, fontweight='bold', y=0.98)
plt.savefig('Figure_1_Russia2025.png', bbox_inches='tight')
plt.close(fig1)
fig2 = plt.figure(figsize=(15.5, 6.8), dpi=300)
plt.subplots_adjust(wspace=0.26)
ax1 = fig2.add_subplot(1, 2, 1)
rel_inc_04 = df_1904['Mean_Inc'] / df_1904['Mean_Inc'].mean()
rel_inc_25 = df_2025['Mean_Inc_2025'] / df_2025['Mean_Inc_2025'].mean()
ax1.scatter(df_1904['Gini'], rel_inc_04, color='#1f4e78', alpha=0.75, s=55, label='1904 Provinces (N = 50)')
ax1.scatter(df_2025['Gini_2025'], rel_inc_25, color='#c00000', alpha=0.75, s=55, marker='s', label='2025 Regions (N = 87)')
ax1.axhline(1.0, color='black', linestyle='--', lw=1.0, label='Epoch Cross-Regional Mean (= 1.0)')
y_spb04 = float(next(iter(rel_inc_04[df_1904['Province'] == 'Sankt-Peterburg'])))
y_chk25 = float(next(iter(rel_inc_25[df_2025['ShortName'] == 'Chukotka AO'])))
y_msk25 = float(next(iter(rel_inc_25[df_2025['ShortName'] == 'Moscow'])))
y_olo04 = float(next(iter(rel_inc_04[df_1904['Province'] == 'Olonets'])))
ax1.annotate('St. Petersburg (1904)', xy=(0.611, y_spb04), xytext=(0.48, 2.25), arrowprops=dict(arrowstyle='->', color='#1f4e78', lw=0.8), fontsize=8.0, fontweight='bold', color='#1f4e78')
ax1.annotate('Chukotka (2025)', xy=(0.458, y_chk25), xytext=(0.49, 2.85), arrowprops=dict(arrowstyle='->', color='#800000', lw=0.8), fontsize=8.0, fontweight='bold', color='#800000')
ax1.annotate('Moscow (2025)', xy=(0.449, y_msk25), xytext=(0.49, 2.50), arrowprops=dict(arrowstyle='->', color='#800000', lw=0.8), fontsize=8.0, fontweight='bold', color='#800000')
ax1.annotate('Olonets (1904)', xy=(0.169, y_olo04), xytext=(0.18, 1.85), arrowprops=dict(arrowstyle='->', color='#1f4e78', lw=0.8), fontsize=8.0, color='#1f4e78')
ax1.set_xlim(0.14, 0.65)
ax1.set_ylim(0.35, 3.25)
ax1.set_xlabel("Regional Gini Coefficient", fontsize=9.5)
ax1.set_ylabel("Relative Mean Income (Epoch Average = 1.0)", fontsize=9.5)
ax1.set_title("A. Inequality vs. Relative Income Across Two Centuries (1904 vs. 2025)", fontsize=10.2, fontweight='bold')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, linestyle='--', alpha=0.4)
ax2 = fig2.add_subplot(1, 2, 2)
metrics_lbl = ['Spatial Localization' + NL + 'PR_reg(u1)', 'Feature Coherence' + NL + 'PR_feat(v1)', 'Top-2 Regions' + NL + 'Energy Share']
vals_04 = [pru_04, prv_04, 0.639]
vals_25_all = [pru_25, prv_25, 0.284]
top2_eur = float(pd.Series((u1_25e**2)).sort_values(ascending=False).head(2).sum())
vals_25_eur = [pru_25e, prv_25e, top2_eur]
xm = np.arange(len(metrics_lbl))
wm = 0.24
ax2.bar(xm - wm, vals_04, width=wm, color='#1f4e78', label='1904 Imperial Russia (N = 50)')
ax2.bar(xm, vals_25_all, width=wm, color='#c00000', label='2025 Russian Federation (N = 87)')
ax2.bar(xm + wm, vals_25_eur, width=wm, color='#ed7d31', label='2025 European Russia Only (N = 58)')

for x_pos, v04, v25a, v25e in zip(xm, vals_04, vals_25_all, vals_25_eur):
    ax2.text(x_pos - wm, v04 + 0.02, f"{v04:.3f}", ha='center', fontsize=7.8, fontweight='bold', color='#1f4e78')
    ax2.text(x_pos, v25a + 0.02, f"{v25a:.3f}", ha='center', fontsize=7.8, fontweight='bold', color='#800000')
    ax2.text(x_pos + wm, v25e + 0.02, f"{v25e:.3f}", ha='center', fontsize=7.8, fontweight='bold', color='#9c4a00')

ax2.set_xticks(xm)
ax2.set_xticklabels(metrics_lbl, fontsize=8.8)
ax2.set_ylim(0, 1.12)
ax2.set_ylabel("Normalized Metric Value (0 to 1 Scale)", fontsize=9.5)
ax2.set_title("B. Diffusion of Capital Dualism: 1904 vs. 2025 (Full & European)", fontsize=10.2, fontweight='bold')
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(True, axis='y', linestyle='--', alpha=0.4)

fig2.suptitle("From Two Capitals to a Resource-Capital Archipelago: Structural Evolution of Russian Inequality (1904-2025)", fontsize=12.0, fontweight='bold', y=1.01)
plt.savefig('Figure_2_Russia2025.png', bbox_inches='tight')
plt.close(fig2)
print("Clean two-century pipeline completed!")


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


=== TWO-CENTURY SPATIAL DUALISM AUDIT (1904 vs 2025) ===
Imperial Russia (1904)           | N=50 | Var1=73.8% | PR_reg=0.072 [0.040, 0.202] | PR_feat=0.802 | Top2=63.9%
Modern Russia All (2025)         | N=87 | Var1=89.1% | PR_reg=0.161 [0.120, 0.195] | PR_feat=0.989 | Top2=28.4%
Modern European RF (2025)        | N=58 | Var1=87.5% | PR_reg=0.131 [0.103, 0.269] | PR_feat=0.984 | Top2=44.9%
Clean two-century pipeline completed!
